#### Initialize Context & Seed Ledger

In [4]:
import sys
from pathlib import Path
HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path: sys.path.insert(0, str(PARENT))

from server.scripts.h3.h3_load import load_geodf_from_csv
PROGRESS_LEDGER = Path("../../map/progress_ledger.csv")
if not PROGRESS_LEDGER.parent.exists():
    raise FileNotFoundError(f"PROGRESS_LEDGER Not Found.")
else:
    print("Loading existing progress ledger")
    progress_geodf = load_geodf_from_csv(PROGRESS_LEDGER)
    print(f"Total Seed Cells: {len(progress_geodf)}")

Loading existing progress ledger
Total Seed Cells: 726


#### Select Cells

In [5]:
SELECTED_CELLS = []
COORDINATES = [51.514340, -0.108365, 20000]
ID_FIELD = "tile_id"
from server.scripts.h3.h3_selecttiles import select_tiles_in_radius
from server.scripts.h3.h3_visualizemap import visualize_progress

if not SELECTED_CELLS:
    seed_select = select_tiles_in_radius(progress_geodf, COORDINATES[0], COORDINATES[1], COORDINATES[2])
    SELECTED_CELLS = seed_select[ID_FIELD].tolist()

existing_cells = progress_geodf[(progress_geodf['scrubbed']==True)][ID_FIELD].tolist()
SELECTED_CELLS = [CELL for CELL in SELECTED_CELLS if CELL not in existing_cells]

progress_geodf["current"] = False
progress_geodf["current"] = progress_geodf[ID_FIELD].apply(lambda x: x in SELECTED_CELLS)
print(f"Selected {progress_geodf['current'].sum()} cells for Nearby Search.")

Selected 726 cells for Nearby Search.


In [6]:
from server.scripts.h3.h3_load import load_boundary_from_json

print("Updating Ledger")
progress_geodf.to_csv(PROGRESS_LEDGER, index=False)
inner_union = load_boundary_from_json(json_path='../get_seed_map_level1/boundary.json')
visualize_progress(progress_geodf, inner_union, output_path=PROGRESS_LEDGER.with_suffix(".html"))

Updating Ledger
Saved Progress Map to: ..\..\map\progress_ledger.html


#### Run APIs

In [7]:
ENABLE_API = True
from server.scripts.get_places.get_places_by_cell import get_places_by_cell
selected_cells = progress_geodf[progress_geodf['current'] == True]
res_geodf = await get_places_by_cell(selected_cells, out_path=Path("../../out/places_cache"))

API calls executed for 0-1: 1 | failures: 0 | places: 15
API calls executed for 0-0: 2 | failures: 0 | places: 22
API calls executed for 0-3: 3 | failures: 0 | places: 32
API calls executed for 0-6: 4 | failures: 0 | places: 33
API calls executed for 0-2: 5 | failures: 0 | places: 42
API calls executed for 0-5: 6 | failures: 0 | places: 58
API calls executed for 0-4: 7 | failures: 0 | places: 64
API calls executed for 1-0: 8 | failures: 0 | places: 65
API calls executed for 1-1: 9 | failures: 0 | places: 70
API calls executed for 1-2: 10 | failures: 0 | places: 73
API calls executed for 1-6: 11 | failures: 0 | places: 76
API calls executed for 1-3: 12 | failures: 0 | places: 86
API calls executed for 1-4: 13 | failures: 0 | places: 89
API calls executed for 2-0: 14 | failures: 0 | places: 95
API calls executed for 2-1: 15 | failures: 0 | places: 105
API calls executed for 2-6: 16 | failures: 0 | places: 112
API calls executed for 2-5: 17 | failures: 0 | places: 116
API calls executed f

In [10]:
cell_geodf = res_geodf.copy()
cell_geodf["tile_id"] = res_geodf["tile_id"].astype(str)
cell_geodf["seed_index"] = res_geodf["seed_index"].astype(int)
cell_geodf["tile_path_id"] = res_geodf["tile_path_id"].astype(str)

#### File API Response and Update Ledger

In [11]:
cell_scrubbed = cell_geodf[cell_geodf['scrubbed'] == True].copy()
if not cell_scrubbed.empty:
    print("Fetch Complete. Updating Ledger.")
    
    # Normalize key type before matching rows between dataframes.
    progress_geodf['tile_id'] = progress_geodf['tile_id'].astype(str)
    cell_scrubbed['tile_id'] = cell_scrubbed['tile_id'].astype(str)

    success_ids = cell_scrubbed['tile_id'].dropna()
    progress_geodf.loc[progress_geodf['tile_id'].isin(success_ids), 'scrubbed'] = True
    
    # Propagate per-tile places_count from this batch into the progress ledger.
    places_count_by_id = (
        cell_scrubbed
        .drop_duplicates(subset=['tile_id'], keep='last')
        .set_index('tile_id')['places_count']
    )
    matched_mask = progress_geodf['tile_id'].isin(places_count_by_id.index)
    progress_geodf.loc[matched_mask, 'places_count'] = progress_geodf.loc[matched_mask, 'tile_id'].map(places_count_by_id)
    
    progress_geodf['current'] = False
    progress_geodf.to_csv(PROGRESS_LEDGER, index=False)
    visualize_progress(progress_geodf, inner_union, output_path=PROGRESS_LEDGER.with_suffix(".html"))
else:
    print("Response InValid")

Fetch Complete. Updating Ledger.
Saved Progress Map to: ..\..\map\progress_ledger.html
